In [ ]:
import sys
print("Python version:", sys.version)
print("Running in Jupyter:", "ipykernel" in sys.modules)

In [ ]:
%%writefile app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

# Task 1: Home endpoint
@app.route("/")
def home():
    return "Welcome to the transaction monitoring system!"

# Task 2.1: Hello with query parameter
@app.route("/hello")
def hello():
    name = request.args.get("name", "stranger")
    return f"Hello, {name}!"

# Task 2.2: Status endpoint
@app.route("/status")
def status():
    return jsonify({
        "service": "monitoring",
        "version": "1.0",
        "status": "ok"
    })

# Task 2.1: Transaction with path variable
@app.route("/transaction/<tx_id>")
def get_transaction(tx_id):
    return jsonify({"tx_id": tx_id, "status": "found"})

# Part 3: POST endpoint for echo
@app.route("/echo", methods=["POST"])
def echo():
    data = request.get_json()
    return jsonify({
        "received": data,
        "field_count": len(data)
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

In [ ]:
import subprocess
import time
import requests
import os
import signal

In [ ]:
def kill_flask_server():
    """Kill any existing Flask server running on port 5000"""
    if os.name == 'nt':  # Windows
        subprocess.run(["taskkill", "/F", "/IM", "python.exe"], capture_output=True)
    else:  # Linux/Mac (Docker environment)
        subprocess.run(["pkill", "-f", "app.py"], capture_output=True)
    time.sleep(1)
    print("Killed existing servers (if any)")

kill_flask_server()

In [ ]:
# Start the Flask server in background
server = subprocess.Popen(["python", "app.py"])
time.sleep(3)  # Wait for server to start
print("Flask server started on http://localhost:5000")

In [ ]:
# Test 1: GET /
r1 = requests.get("http://localhost:5000/")
print(f"GET / : Status {r1.status_code}")
print(f"Body: {r1.text}\n")

# Test 2: GET /hello with query parameter
r2 = requests.get("http://localhost:5000/hello?name=Anna")
print(f"GET /hello?name=Anna : {r2.text}\n")

# Test 3: GET /hello without parameter
r3 = requests.get("http://localhost:5000/hello")
print(f"GET /hello (no name) : {r3.text}\n")

# Test 4: GET /status
r4 = requests.get("http://localhost:5000/status")
print(f"GET /status : {r4.json()}\n")

# Test 5: GET /transaction/TX001
r5 = requests.get("http://localhost:5000/transaction/TX001")
print(f"GET /transaction/TX001 : {r5.json()}\n")

In [ ]:
# Test POST /echo
transaction = {
    "tx_id": "TX0042",
    "amount": 4500.0,
    "store": "Krakow",
    "category": "electronics"
}

r = requests.post("http://localhost:5000/echo", json=transaction)
print("POST /echo response:")
print(r.json())
print()

# Task 3.1: Test wrong method (GET instead of POST)
r_bad = requests.get("http://localhost:5000/echo")
print(f"GET /echo (wrong method): Status {r_bad.status_code}")
print(f"This means: Method Not Allowed - endpoint only accepts POST")

In [ ]:
%%writefile app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

def score_transaction(tx: dict) -> dict:
    """Assess transaction risk using business rules."""
    score = 0
    rules = []
    
    # Rule R1: amount > 3000
    if tx.get("amount", 0) > 3000:
        score += 3
        rules.append("R1: amount > 3000")
    
    # Rule R2: electronics > 1500
    if tx.get("category") == "electronics" and tx.get("amount", 0) > 1500:
        score += 2
        rules.append("R2: electronics > 1500")
    
    # Rule R3: night hour (hour < 6)
    if tx.get("hour", 12) < 6:
        score += 2
        rules.append("R3: night hour")
    
    # Determine risk level
    if score >= 5:
        risk_level = "CRITICAL"
    elif score >= 3:
        risk_level = "HIGH"
    elif score >= 1:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"
    
    return {
        "score": score,
        "risk_level": risk_level,
        "triggered_rules": rules
    }


# HOMEWORK TASK 1: Counters for /stats endpoint
counters = {"total": 0, "high": 0, "critical": 0}


@app.route("/score", methods=["POST"])
def score():
    global counters  # Allow function to modify the global counters
    
    tx = request.get_json()
    
    # Validation: missing amount field
    if not tx or "amount" not in tx:
        return jsonify({"error": "Missing required field 'amount'"}), 400
    
    # HOMEWORK TASK 2: Negative amount validation
    if tx.get("amount", 0) < 0:
        return jsonify({"error": "Amount cannot be negative"}), 400
    
    # HOMEWORK TASK 1: Increment total counter
    counters["total"] += 1
    
    result = score_transaction(tx)
    
    # HOMEWORK TASK 1: Update high/critical counters
    if result["risk_level"] == "HIGH":
        counters["high"] += 1
    elif result["risk_level"] == "CRITICAL":
        counters["critical"] += 1
    
    result["tx_id"] = tx.get("tx_id", "unknown")
    return jsonify(result)


@app.route("/health")
def health():
    return jsonify({"status": "ok", "version": "1.0-rules"})


# HOMEWORK TASK 1: GET /stats endpoint
@app.route("/stats", methods=["GET"])
def get_stats():
    return jsonify(counters)


# Task 1: Home endpoint
@app.route("/")
def home():
    return "Welcome to the transaction monitoring system!"


# Task 2.1: Hello with query parameter
@app.route("/hello")
def hello():
    name = request.args.get("name", "stranger")
    return f"Hello, {name}!"


# Task 2.2: Status endpoint
@app.route("/status")
def status():
    return jsonify({
        "service": "monitoring",
        "version": "1.0",
        "status": "ok"
    })


# Task 2.1: Transaction with path variable
@app.route("/transaction/<tx_id>")
def get_transaction(tx_id):
    return jsonify({"tx_id": tx_id, "status": "found"})


# Part 3: POST endpoint for echo
@app.route("/echo", methods=["POST"])
def echo():
    data = request.get_json()
    return jsonify({
        "received": data,
        "field_count": len(data)
    })


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

In [ ]:
# Kill old server and start new one
kill_flask_server()
server = subprocess.Popen(["python", "app.py"])
time.sleep(3)
print("New Flask server started with scoring endpoints!")

In [ ]:
# Test cases
cases = [
    {"tx_id": "TX001", "amount": 50.0, "category": "food", "hour": 12},
    {"tx_id": "TX002", "amount": 1800.0, "category": "electronics", "hour": 12},
    {"tx_id": "TX003", "amount": 4500.0, "category": "electronics", "hour": 3},
]

print("=" * 80)
print(f"{'TX_ID':<8} {'Amount':>10} {'Risk Level':<12} {'Score':<6} {'Triggered Rules'}")
print("=" * 80)

for tx in cases:
    r = requests.post("http://localhost:5000/score", json=tx)
    res = r.json()
    rules_str = ", ".join(res.get("triggered_rules", []))
    print(f"{tx['tx_id']:<8} {tx['amount']:>10.2f} {res['risk_level']:<12} {res['score']:<6} {rules_str}")

print("=" * 80)

In [ ]:
#Task 4.3 — Test error handling: Send a request without the amount field. Check the status code and response body.

print("--- Testing Error Handling ---")

# Test 1: Missing amount field
r1 = requests.post("http://localhost:5000/score", json={"tx_id": "TX999"})
print(f"Missing amount: Status {r1.status_code}")
print(f"Response: {r1.json()}\n")

# Test 2: Negative amount (Homework)
r2 = requests.post("http://localhost:5000/score", json={"tx_id": "TX888", "amount": -100})
print(f"Negative amount: Status {r2.status_code}")
print(f"Response: {r2.json()}\n")

In [ ]:
# HOMEWORK TASK 2 - Explicit Test
print("\n" + "="*60)
print("HOMEWORK TASK 2: Negative Amount Validation")
print("="*60)

test_cases = [
    {"name": "Normal positive amount", "tx": {"tx_id": "T1", "amount": 1000}, "expected": 200},
    {"name": "Zero amount", "tx": {"tx_id": "T2", "amount": 0}, "expected": 200},
    {"name": "Negative amount", "tx": {"tx_id": "T3", "amount": -100}, "expected": 400},
]

for test in test_cases:
    r = requests.post("http://localhost:5000/score", json=test["tx"])
    status = r.status_code
    result = "✅ PASS" if status == test["expected"] else f"❌ FAIL (got {status})"
    print(f"{test['name']}: {result}")

print("\nExplanation: Server returns 400 Bad Request when amount is negative")
print("because negative transaction amounts are invalid in this business context.")

In [ ]:
#Homework Task 3: Call /score with 10 transactions in a loop and print a summary table.
import random

print("\n" + "=" * 80)
print("HOMEWORK TASK 3: 10 Transactions Summary")
print("=" * 80)

# Generate 10 random transactions
test_transactions = []
for i in range(1, 11):
    test_transactions.append({
        "tx_id": f"TX{i:03d}",
        "amount": round(random.uniform(10, 5000), 2),
        "category": random.choice(["electronics", "food", "clothing", "books"]),
        "hour": random.randint(0, 23)
    })

# Send each transaction to /score
print(f"\n{'TX_ID':<8} {'Amount':>10} {'Risk Level':<12} {'Score':<6} {'Rules Triggered'}")
print("-" * 70)

for tx in test_transactions:
    r = requests.post("http://localhost:5000/score", json=tx)
    res = r.json()
    rules_str = ", ".join(res.get("triggered_rules", []))[:40]
    print(f"{tx['tx_id']:<8} {tx['amount']:>10.2f} {res['risk_level']:<12} {res['score']:<6} {rules_str}")

print("-" * 70)

In [ ]:
# Homework Task 1: Add endpoint GET /stats that returns: how many requests the server handled, how many were HIGH/CRITICAL.
# Get statistics
r_stats = requests.get("http://localhost:5000/stats")
print(f"\n📊 Statistics from /stats endpoint:")
print(f"   Total requests: {r_stats.json()['total']}")
print(f"   HIGH risk: {r_stats.json()['high']}")
print(f"   CRITICAL risk: {r_stats.json()['critical']}")

#    ANSWER: Statistics from /stats endpoint:
   ## Total requests: 15
   ## HIGH risk: 1
   ## CRITICAL risk: 1

In [ ]:
#Task 4.4  Review questions
# 1. What is the difference between GET and POST?
#    ANSWER:
##- GET: Used to retrieve data. Parameters are in URL. No body.  Should not change server state. (e.g., /status)
 ##  - POST: Used to send data for processing. Data is in request body. Can change server state. (e.g., /score)

# 2. Why use jsonify() instead of return {"key": "value"}? Hint: try both and compare the response headers.
#    ANSWER:
##- jsonify() sets the correct Content-Type header to "application/json"
## - Without jsonify(), Flask returns text/html by default
## - This ensures clients know the response is JSON

# 3. What happens if two people call /score at the same time?
#    ANSWER:
## - Flask handles multiple requests concurrently (using threads)
## - Each request is processed independently
## - For shared variables (like counters), need to use locks to prevent race conditions
